In [ ]:
# Slide chunking and Word2vec Embedding with similarity Evaluation

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

In [ ]:
#Load and preview Data
## Step 1: Load and Preview the Dataset
#This step loads the cleaned text data from merged_cleaned_text_PaddleOCR2.csv for processing.

In [3]:

df = pd.read_csv(r"C:\Users\manjo\Downloads\merged_cleaned_text_PaddleOCR2.csv")

# Preview it
df.head()

,filename,text
0,0_0.txt,[ WIKIPEDIA Donate Create account Log in. The ...
1,0_1.txt,[ Animation [edit] Year Title Role Notes 1997 ...
2,0_10.txt,"[ This page was last edited on 20 April 2025, ..."
3,0_100.txt,[ Billy Joel's parents met in the late 1930s a...
4,0_1000.txt,[ Other early instances of women's suffrage in...


In [ ]:
## Step 2: Text Preprocessing
#Cleaning the text by removing brackets and unwanted characters for better chunking and embedding.

In [4]:
def preprocess_text(text):
    if isinstance(text, str) and text.startswith("[") and text.endswith("]"):
        text = text[1:-1].replace("'", "").replace('"', '')
        return " ".join(text.split(","))
    return text

df['processed_text'] = df['text'].apply(preprocess_text)

In [ ]:
## Step 3: Apply Sliding Window Chunking
#Breaking down long texts into overlapping chunks using a sliding window (window size = 50, step size = 25).

In [5]:
def sliding_window_chunk(text, window_size=50, step_size=25):
    words = str(text).split()
    return [
        ' '.join(words[i:i+window_size]) 
        for i in range(0, len(words) - window_size + 1, step_size)
    ]

df['chunks'] = df['processed_text'].apply(lambda x: sliding_window_chunk(x) if pd.notnull(x) else [])

In [6]:
# Combine all chunks from all rows
all_chunks = [chunk for sublist in df['chunks'] for chunk in sublist]

print("Total Chunks:", len(all_chunks))

Total Chunks: 120171


In [7]:
df.to_csv("chunked_output.csv", index=False)

In [ ]:
## Step 4: Flatten Chunked Output
#Flattening the nested list of chunks and saving the result to chunked_text_flat.xlsx

In [8]:
flat_chunks = []

for idx, row in df.iterrows():
    for i, chunk in enumerate(row['chunks']):
        flat_chunks.append({
            'filename': row['filename'],
            'chunk_id': f"{row['filename']}_{i}",
            'chunk_text': chunk
        })

# Create a new DataFrame from the flat list
chunk_df = pd.DataFrame(flat_chunks)

# Save to Excel if needed
chunk_df.to_excel("chunked_text_flat.xlsx", index=False)

# Preview result
chunk_df.head()

,filename,chunk_id,chunk_text
0,0_0.txt,0_0.txt_0,WIKIPEDIA Donate Create account Log in. The Fr...
1,0_0.txt,0_0.txt_1,Herington (born February 22 1988) is a Canadia...
2,0_0.txt,0_0.txt_2,(age 37)[1] Your Mother Good Luck Charlie and ...
3,0_0.txt,0_0.txt_3,show. Occupations Actress singer Big City Gree...
4,0_0.txt,0_0.txt_4,that she voiced. Website marieveherington.com ...


In [16]:
!pip install gensim

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/60.6 kB ? eta -:--:--
     ---------------------------------------- 60.6/60.6 kB 1.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/24.0 MB ? eta -:--:--
   - -------------------------------------- 1.1/24.0 MB 34.4 MB/s eta 0:00:01
   --- ------------------------------------ 2.1/24.0 MB 26.7 MB/s eta 0:00:01
   ----- ---------------------------------- 3.1/24.0 MB 24.5 MB/s eta 0:00:01
   ------- -------------------------------- 4.7/24.0 MB 27.0 MB/s eta 0:00:01
   ----------- ---------------------------- 6.8/24.0 MB 30.8 MB/s eta 0:00:01
   -------------- ------------------------- 8.7/24.0 MB 32.6 MB/s eta 0:00:01
   ----------------- ---------------------- 10.6/24.0 MB 34.6 MB/s eta 0:00:01
   --------------------- ------------------ 13.1/24.0 MB 43.7 MB/s eta 0:00:01
   ---------------------- ----------------- 13.6/24.0 MB 43.7 MB/s eta 0:0

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
## Step 5: Generate Word2Vec Embeddings
#Creating vector representations of each chunk using the Word2Vec model.

In [17]:
import gensim
from gensim.models import Word2Vec
import numpy as np

# Tokenize text
tokenized_chunks = [text.split() for text in chunk_df['chunk_text']]

# Train Word2Vec model (or you can load a pre-trained model)
w2v_model = Word2Vec(sentences=tokenized_chunks, vector_size=100, window=5, min_count=1, workers=4)

# Generate embeddings: average word vectors per chunk
def get_avg_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

chunk_df['w2v_embedding'] = tokenized_chunks
chunk_df['w2v_embedding'] = chunk_df['w2v_embedding'].apply(lambda tokens: get_avg_vector(tokens, w2v_model))

In [18]:
w2v_df = pd.DataFrame(chunk_df['w2v_embedding'].tolist())
w2v_df.insert(0, 'chunk_id', chunk_df['chunk_id'])

In [ ]:
## Step 6: Save Word2Vec Embeddings
##Saving the embeddings in Excel and CSV format for later use and comparison

In [19]:
# Save to Excel or CSV
w2v_df.to_excel("word2vec_embeddings.xlsx", index=False)   # For Excel
w2v_df.to_csv("word2vec_embeddings.csv", index=False)      # For CSV

In [20]:
!pip install rouge-score

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
## Step 7: Evaluate Chunks using Similarity Metrics
#Comparing chunks using:
#Cosine Similarity
#BLEU Score
#ROUGE Score

In [21]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# Load your Word2Vec embeddings
embedding_df = pd.read_csv("word2vec_embeddings.csv")
embedding_df = embedding_df.dropna()
chunk_ids = embedding_df['chunk_id']
X = embedding_df.drop(columns=['chunk_id'])

# 1. Cosine Similarity (sample 5 chunks)
cosine_sim_matrix = cosine_similarity(X[:5])
cosine_df = pd.DataFrame(cosine_sim_matrix, index=chunk_ids[:5], columns=chunk_ids[:5])
print("Cosine Similarity Matrix (Sample):")
print(cosine_df)

# 2. BLEU Score (sample comparisons)
references = [['this', 'is', 'a', 'test'], ['sample', 'document', 'text']]
hypotheses = [['this', 'is', 'test'], ['sample', 'doc', 'text']]

bleu_scores = [
    sentence_bleu([ref], hyp, smoothing_function=SmoothingFunction().method1)
    for ref, hyp in zip(references, hypotheses)
]

print("\nBLEU Scores:")
for i, score in enumerate(bleu_scores):
    print(f"Ref: {references[i]} | Hyp: {hypotheses[i]} | BLEU: {score:.4f}")

# 3. ROUGE Score
rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
rouge_scores = [rouge.score(" ".join(ref), " ".join(hyp)) for ref, hyp in zip(references, hypotheses)]

print("\nROUGE Scores:")
for i, r in enumerate(rouge_scores):
    print(f"Ref: {' '.join(references[i])} | Hyp: {' '.join(hypotheses[i])}")
    print(f"  ROUGE-1: {r['rouge1']}")
    print(f"  ROUGE-L: {r['rougeL']}")

Cosine Similarity Matrix (Sample):
chunk_id   0_0.txt_0  0_0.txt_1  0_0.txt_2  0_0.txt_3  0_0.txt_4
chunk_id                                                        
0_0.txt_0   1.000000   0.875339   0.733545   0.709668   0.685741
0_0.txt_1   0.875339   1.000000   0.868090   0.829443   0.828674
0_0.txt_2   0.733545   0.868090   1.000000   0.957383   0.781524
0_0.txt_3   0.709668   0.829443   0.957383   1.000000   0.863507
0_0.txt_4   0.685741   0.828674   0.781524   0.863507   1.000000

BLEU Scores:
Ref: ['this', 'is', 'a', 'test'] | Hyp: ['this', 'is', 'test'] | BLEU: 0.1905
Ref: ['sample', 'document', 'text'] | Hyp: ['sample', 'doc', 'text'] | BLEU: 0.1351

ROUGE Scores:
Ref: this is a test | Hyp: this is test
  ROUGE-1: Score(precision=1.0, recall=0.75, fmeasure=0.8571428571428571)
  ROUGE-L: Score(precision=1.0, recall=0.75, fmeasure=0.8571428571428571)
Ref: sample document text | Hyp: sample doc text
  ROUGE-1: Score(precision=0.6666666666666666, recall=0.6666666666666666, fmeasure